# Part 3: Vector Database and Semantic Search

In this notebook, we'll:
1. Create a vector database (ChromaDB)
2. Store embeddings and documents
3. Perform semantic search
4. Retrieve relevant documents based on similarity

## Import and Setup

In [ ]:
import sys
sys.path.append('..')

from src.vector_store import VectorStore
from src.embeddings import EmbeddingGenerator
from src.pdf_loader import load_sample_documents
import json

print("✓ Libraries imported")

## Create Vector Store

In [ ]:
# Initialize vector store
store = VectorStore(
    db_path="../chroma_db",
    collection_name="rag_documents"
)

print("✓ Vector store initialized")

# Check current status
info = store.get_collection_info()
print(f"\nCollection info:")
print(json.dumps(info, indent=2))

## Prepare Documents and Embeddings

In [ ]:
# Load sample documents
documents = load_sample_documents()

# Initialize embedding generator
generator = EmbeddingGenerator(embedding_type="local")

# Prepare documents with metadata
docs_to_store = []
texts_to_embed = []

for doc in documents:
    # Split into chunks if document is large
    text = doc['text']
    chunks = [text[i:i+500] for i in range(0, len(text), 500)]
    
    for chunk_idx, chunk in enumerate(chunks):
        docs_to_store.append({
            'text': chunk,
            'metadata': {
                'source': doc['filename'],
                'chunk': chunk_idx,
                'original_pages': doc.get('num_pages', 1)
            }
        })
        texts_to_embed.append(chunk)

print(f"Prepared {len(docs_to_store)} document chunks")

## Generate Embeddings

In [ ]:
# Generate embeddings for all chunks
embeddings = generator.embed_texts(texts_to_embed)

print(f"Generated {len(embeddings)} embeddings")
print(f"Embedding dimension: {len(embeddings[0])}")

## Store in Vector Database

In [ ]:
# Add documents to vector store
store.add_documents(docs_to_store, embeddings)

# Check updated status
info = store.get_collection_info()
print(f"\nVector store updated:")
print(json.dumps(info, indent=2))

## Perform Semantic Search

In [ ]:
# Example queries
queries = [
    "What is machine learning?",
    "Tell me about neural networks",
    "Explain backpropagation"
]

for query in queries:
    # Generate embedding for query
    query_embedding = generator.embed_text(query)
    
    # Search for similar documents
    results = store.search(query_embedding, k=2)
    
    print(f"\nQuery: {query}")
    print("="*60)
    
    if results:
        for i, result in enumerate(results, 1):
            print(f"\nResult {i}:")
            print(f"  Similarity: {result['similarity']:.4f}")
            print(f"  Source: {result['metadata'].get('source', 'Unknown')}")
            print(f"  Text: {result['text'][:100]}...")
    else:
        print("  No results found")

## Analyze Retrieval Results

In [ ]:
# Test with different K values
query = "What is artificial intelligence?"
query_embedding = generator.embed_text(query)

k_values = [1, 2, 3, 5]

for k in k_values:
    results = store.search(query_embedding, k=k)
    print(f"\nK={k}: Retrieved {len(results)} documents")
    
    if results:
        similarities = [r['similarity'] for r in results]
        print(f"  Similarity range: {min(similarities):.4f} - {max(similarities):.4f}")
        print(f"  Average similarity: {sum(similarities)/len(similarities):.4f}")

## Search with Threshold

In [ ]:
# Implement similarity threshold
query = "backpropagation neural network"
query_embedding = generator.embed_text(query)

# Get more results than needed
results = store.search(query_embedding, k=5)

# Filter by similarity threshold
threshold = 0.4
filtered_results = [r for r in results if r['similarity'] >= threshold]

print(f"Query: {query}")
print(f"Total results: {len(results)}")
print(f"Results above threshold ({threshold}): {len(filtered_results)}")
print()

for i, result in enumerate(filtered_results, 1):
    print(f"{i}. Similarity: {result['similarity']:.4f}")
    print(f"   {result['text'][:80]}...\n")

## Analyze Vector Store

In [ ]:
# Get detailed collection statistics
info = store.get_collection_info()

print("Vector Store Statistics:")
print(f"  Collection name: {info['name']}")
print(f"  Document count: {info['document_count']}")
print(f"  Database path: {info['db_path']}")
print(f"  Embedding dimension: {generator.get_embedding_dimension()}")

## Reset and Clear Vector Store

In [ ]:
# Example of clearing the vector store
# Uncomment if you want to start fresh

# store.reset()
# info = store.get_collection_info()
# print("Vector store cleared")
# print(f"Documents: {info['document_count']}")

print("Vector store reset is available but commented out")
print("Uncomment the code above to clear the database")

## Summary

Key concepts from this notebook:
- ✓ ChromaDB is a lightweight vector database
- ✓ Documents are stored with embeddings and metadata
- ✓ Semantic search finds similar documents efficiently
- ✓ K value controls how many results to retrieve
- ✓ Similarity threshold helps filter low-quality results

**Next steps:**
- Continue to notebook 4 to build the complete RAG pipeline
- Or experiment with different queries and K values